# Customer Segmentation for an Online Store (RFM + Clustering)

**Client goal:** group customers by buying behaviour so marketing can target each group
differently (reward the best customers, win back customers who are slipping away, nurture new ones).

**Data:** UCI Online Retail, about 540k transactions from a UK online gift shop, Dec 2010 - Dec 2011
(Kaggle `carrie1/ecommerce-data`).

**Plan:**
1. Clean transactions (cancellations, returns, missing customer IDs, non-product lines)
2. Build an RFM table per customer: **R**ecency, **F**requency, **M**onetary value
3. Choose the number of segments (elbow + silhouette)
4. Compare K-Means, Gaussian Mixture and Agglomerative clustering
5. Profile and name the segments, with a marketing action for each
6. Save the model for the Streamlit app

In [ ]:
import os, subprocess, sys
if os.path.exists("/kaggle"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.7.2"], check=False)

In [ ]:
%%writefile segmentation.py
"""Cleaning, RFM and segment naming shared by the training notebook and the Streamlit app."""
import numpy as np
import pandas as pd

REQUIRED_COLUMNS = ["InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "CustomerID"]
RFM = ["Recency", "Frequency", "Monetary"]
NON_PRODUCT_CODES = {"POST", "D", "M", "BANK CHARGES", "AMAZONFEE", "DOT", "CRUK", "PADS", "C2"}

ACTIONS = {
    "Champions": "Reward them: VIP perks, early access to new products, ask for reviews and referrals.",
    "Loyal Customers": "Upsell and cross-sell, loyalty points, personalised recommendations.",
    "New / Promising": "Onboarding emails, a second-purchase discount, show best-sellers.",
    "At Risk": "Win-back campaign now: personal email, time-limited offer, ask what went wrong.",
    "Lost / Inactive": "Low-cost reactivation (newsletter, big seasonal sale); do not overspend.",
}


def clean_transactions(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    info = {"rows_raw": len(df)}
    df = df.copy()
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)
    info["missing_customer_id"] = int(df["CustomerID"].isna().sum())
    df = df.dropna(subset=["CustomerID"])
    cancel = df["InvoiceNo"].str.startswith("C")
    info["cancellations"] = int(cancel.sum())
    df = df[~cancel]
    bad = (df["Quantity"] <= 0) | (df["UnitPrice"] <= 0)
    info["non_positive_qty_or_price"] = int(bad.sum())
    df = df[~bad]
    if "StockCode" in df:
        non_prod = df["StockCode"].astype(str).str.upper().isin(NON_PRODUCT_CODES)
        info["non_product_lines"] = int(non_prod.sum())
        df = df[~non_prod]
    info["duplicates"] = int(df.duplicated().sum())
    df = df.drop_duplicates()
    df["CustomerID"] = df["CustomerID"].astype(float).astype(int).astype(str)
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]
    info["rows_clean"] = len(df)
    return df.reset_index(drop=True), info


def compute_rfm(tx: pd.DataFrame, snapshot=None) -> pd.DataFrame:
    """One row per customer. Recency = days since last purchase, measured from `snapshot`
    (default: the day after the last transaction in the data)."""
    snapshot = pd.Timestamp(snapshot) if snapshot is not None else tx["InvoiceDate"].max() + pd.Timedelta(days=1)
    rfm = tx.groupby("CustomerID").agg(
        Recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum"),
    )
    return rfm


def name_segments(profile: pd.DataFrame) -> dict:
    """Give each cluster a business name from its median Recency/Frequency/Monetary.
    profile: index = cluster id, columns = RFM medians; overall medians in profile.attrs."""
    ranks = (profile["Recency"].rank(ascending=False) + profile["Frequency"].rank()
             + profile["Monetary"].rank())
    order = ranks.sort_values(ascending=False).index.tolist()
    med = profile.attrs["overall_median"]
    names = {order[0]: "Champions", order[-1]: "Lost / Inactive"}
    for c in order[1:-1]:
        recent = profile.loc[c, "Recency"] <= med["Recency"]
        frequent = profile.loc[c, "Frequency"] > med["Frequency"]
        if recent and frequent:
            name = "Loyal Customers"
        elif recent:
            name = "New / Promising"
        else:
            name = "At Risk"
        while name in names.values():  # keep names unique if two clusters look alike
            name += " (2)"
        names[c] = name
    return names


def action_for(name: str) -> str:
    return ACTIONS.get(name.replace(" (2)", ""), "")

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from segmentation import RFM, action_for, clean_transactions, compute_rfm, name_segments

SEED = 42
for d in ["data", "model", "reports"]:
    Path(d).mkdir(exist_ok=True)
print("scikit-learn", sklearn.__version__)


def find(name):
    for base in [Path("/kaggle/input"), Path("data")]:
        hits = list(base.rglob(name)) if base.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found; attach the dataset or put it in data/")

## 1. Load and clean

In [ ]:
raw = pd.read_csv(find("data.csv"), encoding="ISO-8859-1", dtype={"CustomerID": "float64"})
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], format="%m/%d/%Y %H:%M")
print(raw.shape)
raw.head()

In [ ]:
tx, clean_info = clean_transactions(raw)
print(json.dumps(clean_info, indent=2))
print(f"{tx['CustomerID'].nunique()} customers, {tx['InvoiceNo'].nunique()} orders, "
      f"{tx['InvoiceDate'].min().date()} to {tx['InvoiceDate'].max().date()}")

## 2. RFM table

In [ ]:
rfm = compute_rfm(tx)
rfm.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, RFM):
    ax.hist(rfm[col], bins=50)
    ax.set_title(col)
    ax.set_yscale("log")
fig.suptitle("RFM distributions (log count): heavily skewed, so log-transform before clustering")
fig.tight_layout()
fig.savefig("reports/rfm_distributions.png", dpi=110)
plt.show()

All three are heavily right-skewed (a few wholesale customers spend 100x the median), so the
features are **log-transformed and standardised** before clustering. Otherwise, a handful of
huge buyers would decide the clusters.

## 3. How many segments?

In [ ]:
prep = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler())
Z = prep.fit_transform(rfm[RFM])

k_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
    k_rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Z, km.labels_),
                   "davies_bouldin": davies_bouldin_score(Z, km.labels_)})
k_scan = pd.DataFrame(k_rows).round(4)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
k_scan.plot(x="k", y="inertia", marker="o", ax=axes[0], title="Elbow (inertia)", legend=False)
k_scan.plot(x="k", y="silhouette", marker="o", ax=axes[1], title="Silhouette (higher is better)",
            legend=False)
fig.tight_layout()
fig.savefig("reports/choose_k.png", dpi=110)
plt.show()
k_scan

Two clusters often score the best silhouette, but "good vs bad customers" is too coarse for
marketing. We pick the best silhouette among **4 to 6 segments**, which are still easy for a
marketing team to act on.

In [ ]:
K = int(k_scan[k_scan.k.between(4, 6)].sort_values("silhouette", ascending=False).iloc[0]["k"])
print("Chosen number of segments:", K)

## 4. Compare clustering algorithms (same K)
Agglomerative clustering cannot assign **new** customers without refitting, so it is shown as a
reference only. The deployed model is chosen between K-Means and Gaussian Mixture.

In [ ]:
algos = {
    "K-Means": KMeans(n_clusters=K, n_init=20, random_state=SEED),
    "Gaussian Mixture": GaussianMixture(n_components=K, n_init=5, random_state=SEED),
    "Agglomerative (Ward)": AgglomerativeClustering(n_clusters=K, linkage="ward"),
}
rows, labels = [], {}
for name, algo in algos.items():
    lab = algo.fit_predict(Z)
    labels[name] = lab
    sizes = np.bincount(lab)
    rows.append({"model": name, "silhouette": silhouette_score(Z, lab),
                 "davies_bouldin": davies_bouldin_score(Z, lab),
                 "calinski_harabasz": calinski_harabasz_score(Z, lab),
                 "smallest_segment_pct": sizes.min() / len(lab),
                 "can_score_new_customers": name != "Agglomerative (Ward)"})
results = pd.DataFrame(rows).round(4)
deployable = results[results.can_score_new_customers]
best = deployable.sort_values("silhouette", ascending=False).iloc[0]["model"]
results.to_csv("reports/model_comparison.csv", index=False)
print("Selected:", best)
results

Silhouette and Calinski-Harabasz: higher is better. Davies-Bouldin: lower is better.

## 5. Profile and name the segments

In [ ]:
final = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler(),
                      KMeans(n_clusters=K, n_init=20, random_state=SEED) if best == "K-Means"
                      else GaussianMixture(n_components=K, n_init=5, random_state=SEED))
final.fit(rfm[RFM])
rfm["cluster"] = final.predict(rfm[RFM])

profile = rfm.groupby("cluster")[RFM].median()
profile.attrs["overall_median"] = rfm[RFM].median().to_dict()
names = {int(k): v for k, v in name_segments(profile).items()}
rfm["Segment"] = rfm["cluster"].map(names)

summary = rfm.groupby("Segment").agg(
    customers=("Recency", "size"), median_recency_days=("Recency", "median"),
    median_orders=("Frequency", "median"), median_spend=("Monetary", "median"),
    total_revenue=("Monetary", "sum"))
summary["customers_pct"] = summary["customers"] / summary["customers"].sum()
summary["revenue_pct"] = summary["total_revenue"] / summary["total_revenue"].sum()
summary["action"] = [action_for(s) for s in summary.index]
summary = summary.sort_values("median_spend", ascending=False).round(3)
summary

In [ ]:
colors = dict(zip(summary.index, plt.cm.tab10.colors))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for seg, g in rfm.groupby("Segment"):
    axes[0].scatter(g["Recency"], g["Monetary"], s=6, alpha=0.5, label=seg, color=colors[seg])
axes[0].set(yscale="log", xlabel="Recency (days since last order)", ylabel="Total spend (log)",
            title="Customers by segment")
axes[0].legend(fontsize=8, markerscale=3)
summary["customers_pct"].plot.bar(ax=axes[1], rot=20, color=[colors[s] for s in summary.index],
                                  title="Share of customers")
summary["revenue_pct"].plot.bar(ax=axes[2], rot=20, color=[colors[s] for s in summary.index],
                                title="Share of revenue")
for ax in axes[1:]:
    ax.tick_params(labelsize=8)
fig.tight_layout()
fig.savefig("reports/segments_overview.png", dpi=110)
plt.show()

## 6. Save model and outputs for the app

In [ ]:
joblib.dump({"pipeline": final, "names": names}, "model/best_model.joblib", compress=3)
rfm.reset_index().to_csv("data/customer_segments.csv", index=False)
summary.to_csv("reports/segment_summary.csv")

# Small sample of raw transactions so the app's upload feature can be tried
sample_ids = rfm.groupby("Segment").sample(n=40, random_state=SEED, replace=True).index.unique()
cols = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID",
        "Country"]
tx[tx["CustomerID"].isin(sample_ids)][cols].to_csv("data/sample_transactions.csv", index=False)

meta = {
    "selected_model": best, "n_segments": K,
    "selection_rule": "best silhouette among models that can score new customers, K chosen in 4-6",
    "sklearn_version": sklearn.__version__, "clean_info": clean_info,
    "customers": int(len(rfm)), "snapshot_date": str((tx["InvoiceDate"].max() + pd.Timedelta(days=1)).date()),
    "results": results.to_dict(orient="records"), "k_scan": k_scan.to_dict(orient="records"),
    "segment_names": names,
}
Path("model/metrics.json").write_text(json.dumps(meta, indent=2, default=str))


def md_table(df, index=False):
    df = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(f"{v:,}" if isinstance(v, float) else str(v)
                                              for v in row) + " |" for row in df.values)


ci = clean_info
report = f"""# Evaluation Report: Customer Segmentation

## Data and cleaning
- UCI Online Retail: {ci['rows_raw']:,} transaction lines, {meta['snapshot_date']} snapshot.
- Removed: {ci['missing_customer_id']:,} lines without a customer ID, {ci['cancellations']:,} cancellations,
  {ci['non_positive_qty_or_price']:,} lines with zero/negative quantity or price,
  {ci.get('non_product_lines', 0):,} postage/fee lines, {ci['duplicates']:,} duplicates.
- {ci['rows_clean']:,} clean lines -> {meta['customers']:,} customers.

## Method
- RFM per customer: Recency (days since last order), Frequency (number of orders), Monetary (total spend).
- log(1+x) transform and standard scaling (the raw values are extremely skewed).
- Number of segments: K-Means scan of k = 2..8 (elbow, silhouette, Davies-Bouldin); best silhouette
  among 4-6 segments -> **K = {K}**.
- Algorithms compared at K = {K}: K-Means, Gaussian Mixture, Agglomerative (Ward).

## Results
{md_table(results)}

**Selected: {best}** (best silhouette among models that can score new customers).

### Segments
{md_table(summary, index=True)}

![Segments](segments_overview.png)
![Choosing k](choose_k.png)

## Limitations
- Clustering has no "correct answer"; the metrics show the groups are reasonably separated, and the
  segment names are rules on the group medians that marketing should review.
- One year of data from one UK gift retailer (many customers are wholesalers); segments should be
  refitted on the client's own customers.
- Recency depends on the snapshot date; re-score customers regularly (e.g. monthly).
- RFM ignores what customers buy, discounts and returns; product-category features could be added later.
"""
Path("reports/evaluation_report.md").write_text(report, encoding="utf-8")
print(report)